# 05 — Transportabilidad geográfica interna (dejar-una-entidad-fuera)

**Tesis:** Estratificación de riesgo clínico consciente de la equidad — sesgo geográfico entre entidades.

La partición aleatoria de los notebooks 02–04 reparte cada entidad entre entrenamiento y prueba, de modo que el modelo **siempre ha visto** casos del estado en el que se le evalúa. Eso mide interpolación dentro de una población conocida, no generalización a un contexto nuevo.

Este notebook entrena 32 modelos: cada uno excluye por completo una entidad y se evalúa sobre ella. Es el diseño que corresponde a la pregunta central de la tesis —¿el modelo funciona igual donde no fue entrenado?— y cuantifica la **pérdida por transportabilidad** entidad por entidad.

> **Alcance.** Esto **no** pone a prueba la hipótesis H4, que habla de una *fuente independiente*. Aquí la fuente, el año y el proceso de captura son los mismos; lo único que cambia es que la entidad evaluada no participó en el entrenamiento. H4 requiere la base de Egresos Hospitalarios de la DGIS y sigue pendiente. Lo que sí se somete a prueba es la generalización geográfica, que es el eje del trabajo.

## 1. Datos y configuración

In [1]:
import json

import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

import tesis_common as tc

# Se reentrena deliberadamente, a diferencia de los notebooks 03 y 04: el objeto
# de estudio ES el modelo que nunca vio la entidad evaluada. Los hiperparámetros
# son los canónicos, de modo que la única diferencia con la línea de base es qué
# entidades entraron al entrenamiento.
df, manifiesto = tc.cargar_cohorte()
X = df[tc.PREDICTORES].astype("float32")
y = df[tc.TARGET].astype(int)
ent = df["entidad_nombre"]

ENTIDADES = sorted(ent.dropna().unique())
print("Entidades a evaluar:", len(ENTIDADES))
print("Hiperparámetros:", tc.HPARAMS_HGB)
print("Sensibilidad objetivo:", tc.SENSIBILIDAD_OBJETIVO)

Entidades a evaluar: 32
Hiperparámetros: {'max_iter': 300, 'learning_rate': 0.05, 'max_depth': 6, 'class_weight': 'balanced', 'random_state': 42}
Sensibilidad objetivo: 0.8


## 2. Referencia interna

Para medir la pérdida por transportabilidad hace falta un punto de comparación: el desempeño **por entidad** del modelo de referencia, que sí vio esa entidad. Se toma del notebook 03.

In [2]:
with open("auditoria_equidad.json", encoding="utf-8") as fh:
    aud = json.load(fh)
interno = {r["grupo"]: r for r in aud["por_entidad"]}
print(f"Referencia interna cargada: {len(interno)} entidades "
      f"| AUROC global {aud['auroc_global']}")

Referencia interna cargada: 32 entidades | AUROC global 0.9503


## 3. Dejar-una-entidad-fuera

Para cada entidad $g$: se entrena con las 31 restantes, se calibra sobre un bloque tallado de esas mismas 31 (nunca de $g$), se fija el umbral que alcanza la sensibilidad objetivo **en las entidades de entrenamiento** —que es la información disponible en despliegue— y se evalúa sobre la totalidad de $g$.

In [3]:
filas = []
for k, g in enumerate(ENTIDADES, 1):
    fuera = (ent == g).to_numpy()
    idx_fuera = np.where(fuera)[0]
    idx_dentro = np.where(~fuera)[0]

    # Bloque de calibración tallado SOLO de las entidades de entrenamiento.
    y_dentro = y.iloc[idx_dentro].to_numpy()
    i_tr, i_cal = train_test_split(idx_dentro, test_size=tc.FRACCION_CALIB,
                                   stratify=y_dentro, random_state=tc.SEMILLA)

    m = HistGradientBoostingClassifier(**tc.HPARAMS_HGB).fit(X.iloc[i_tr], y.iloc[i_tr])
    cal = IsotonicRegression(out_of_bounds="clip").fit(
        m.predict_proba(X.iloc[i_cal])[:, 1], y.iloc[i_cal])

    # Umbral fijado con datos de entrenamiento: en despliegue no se conoce el
    # desenlace de la entidad nueva, así que tomarlo de ella sería hacer trampa.
    p_cal = cal.predict(m.predict_proba(X.iloc[i_cal])[:, 1])
    thr, _, _ = tc.umbral_para_sensibilidad(y.iloc[i_cal].to_numpy(), p_cal)

    p_fuera = cal.predict(m.predict_proba(X.iloc[idx_fuera])[:, 1])
    y_fuera = y.iloc[idx_fuera].to_numpy()
    tpr, fpr = tc.tasas(y_fuera, p_fuera, thr)

    filas.append({
        "entidad": g, "n": int(len(idx_fuera)), "tasa": float(y_fuera.mean()),
        "AUROC_loeo": float(roc_auc_score(y_fuera, p_fuera)),
        "AUPRC_loeo": float(average_precision_score(y_fuera, p_fuera)),
        "brier_loeo": float(brier_score_loss(y_fuera, p_fuera)),
        "TPR_loeo": float(tpr), "FPR_loeo": float(fpr), "umbral": float(thr),
        "riesgo_medio": float(p_fuera.mean()),
        "AUROC_interno": float(interno[g]["AUROC"]),
        "TPR_interno": float(interno[g]["TPR"]),
    })
    print(f"[{k:2d}/32] {g:20s} AUROC {filas[-1]['AUROC_loeo']:.4f} "
          f"(interno {filas[-1]['AUROC_interno']:.4f})", flush=True)

L = pd.DataFrame(filas)
L["delta_AUROC"] = L["AUROC_loeo"] - L["AUROC_interno"]
L["delta_TPR"] = L["TPR_loeo"] - L["TPR_interno"]
print("\nListo.")

[ 1/32] AGUASCALIENTES       AUROC 0.9318 (interno 0.9304)


[ 2/32] BAJA CALIFORNIA      AUROC 0.9585 (interno 0.9584)


[ 3/32] BAJA CALIFORNIA SUR  AUROC 0.9292 (interno 0.9272)


[ 4/32] CAMPECHE             AUROC 0.9601 (interno 0.9544)


[ 5/32] CHIAPAS              AUROC 0.9283 (interno 0.9265)


...

[29/32] TLAXCALA             AUROC 0.9283 (interno 0.9391)


[30/32] VERACRUZ             AUROC 0.9314 (interno 0.9340)


[31/32] YUCATAN              AUROC 0.9480 (interno 0.9440)


[32/32] ZACATECAS            AUROC 0.9421 (interno 0.9431)



Listo.


## 4. ¿Se transfiere el desempeño?

In [4]:
print("== PERDIDA POR TRANSPORTABILIDAD (AUROC fuera de entidad - AUROC interno) ==")
print(f"  mediana {L['delta_AUROC'].median():+.4f} | media {L['delta_AUROC'].mean():+.4f}")
print(f"  rango   [{L['delta_AUROC'].min():+.4f}, {L['delta_AUROC'].max():+.4f}]")
print(f"  entidades que empeoran: {int((L['delta_AUROC'] < 0).sum())} de {len(L)}")

print("\n== LAS 5 QUE MAS PIERDEN ==")
print(L.nsmallest(5, "delta_AUROC")[
    ["entidad", "n", "tasa", "AUROC_interno", "AUROC_loeo", "delta_AUROC"]]
    .round(4).to_string(index=False))
print("\n== LAS 5 QUE MENOS ==")
print(L.nlargest(5, "delta_AUROC")[
    ["entidad", "n", "tasa", "AUROC_interno", "AUROC_loeo", "delta_AUROC"]]
    .round(4).to_string(index=False))

== PERDIDA POR TRANSPORTABILIDAD (AUROC fuera de entidad - AUROC interno) ==
  mediana -0.0006 | media -0.0007
  rango   [-0.0108, +0.0066]
  entidades que empeoran: 19 de 32

== LAS 5 QUE MAS PIERDEN ==
   entidad      n   tasa  AUROC_interno  AUROC_loeo  delta_AUROC
  TLAXCALA  17431 0.0735         0.9391      0.9283      -0.0108
 MICHOACAN  38126 0.1285         0.9359      0.9280      -0.0079
   MORELOS  39258 0.0758         0.9600      0.9552      -0.0048
    PUEBLA  75074 0.1111         0.9480      0.9435      -0.0045
GUANAJUATO 118324 0.0574         0.9579      0.9539      -0.0040

== LAS 5 QUE MENOS ==
   entidad     n   tasa  AUROC_interno  AUROC_loeo  delta_AUROC
   DURANGO 25123 0.0544         0.9310      0.9376       0.0066
  CAMPECHE 16881 0.0631         0.9544      0.9601       0.0057
TAMAULIPAS 63741 0.0509         0.9539      0.9592       0.0053
   YUCATAN 49748 0.0690         0.9440      0.9480       0.0040
 CHIHUAHUA 44162 0.0717         0.9207      0.9236       0.0029

## 5. ¿Se acentúan las disparidades?

La segunda cláusula de H4 —trasladada aquí a la transportabilidad geográfica interna— pregunta si las brechas crecen cuando el contexto es nuevo. Se comparan las dispersiones entre entidades bajo ambos regímenes.

In [5]:
comp = pd.DataFrame({
    "regimen": ["interno (la entidad estuvo en entrenamiento)",
                "dejar-una-fuera (entidad no vista)"],
    "AUROC_min": [L["AUROC_interno"].min(), L["AUROC_loeo"].min()],
    "AUROC_max": [L["AUROC_interno"].max(), L["AUROC_loeo"].max()],
    "gap_AUROC": [L["AUROC_interno"].max() - L["AUROC_interno"].min(),
                  L["AUROC_loeo"].max() - L["AUROC_loeo"].min()],
    "TPR_min": [L["TPR_interno"].min(), L["TPR_loeo"].min()],
    "TPR_max": [L["TPR_interno"].max(), L["TPR_loeo"].max()],
    "gap_TPR": [L["TPR_interno"].max() - L["TPR_interno"].min(),
                L["TPR_loeo"].max() - L["TPR_loeo"].min()],
}).round(4)
print(comp.to_string(index=False))

from scipy import stats
r_n, p_n = stats.pearsonr(L["delta_AUROC"], np.log10(L["n"]))
r_t, p_t = stats.pearsonr(L["delta_AUROC"], L["tasa"])
print(f"\nCorr(perdida, log10 tamano de la entidad): r={r_n:+.3f} (p={p_n:.3f})")
print(f"Corr(perdida, tasa de mortalidad)        : r={r_t:+.3f} (p={p_t:.3f})")
print("\n-> si la perdida se concentra en entidades pequenas, el modelo depende de")
print("   haber visto ese estado; si no, la generalizacion geografica es robusta.")

                                     regimen  AUROC_min  AUROC_max  gap_AUROC  TPR_min  TPR_max  gap_TPR
interno (la entidad estuvo en entrenamiento)     0.9207     0.9770     0.0563   0.6313   0.9248   0.2935
          dejar-una-fuera (entidad no vista)     0.9236     0.9752     0.0516   0.6660   0.9343   0.2684

Corr(perdida, log10 tamano de la entidad): r=-0.003 (p=0.988)
Corr(perdida, tasa de mortalidad)        : r=-0.370 (p=0.037)

-> si la perdida se concentra en entidades pequenas, el modelo depende de
   haber visto ese estado; si no, la generalizacion geografica es robusta.


## 6. Figura y exportación

In [6]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

orden = L.sort_values("AUROC_loeo")
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
ypos = np.arange(len(orden))
ax[0].barh(ypos - 0.2, orden["AUROC_interno"], height=0.4, color="#2E5496", label="interno")
ax[0].barh(ypos + 0.2, orden["AUROC_loeo"], height=0.4, color="#C00000", label="entidad no vista")
ax[0].set_yticks(ypos); ax[0].set_yticklabels(orden["entidad"], fontsize=7)
ax[0].set_xlim(0.5, 1.0); ax[0].legend(loc="lower right")
ax[0].set_title("AUROC por entidad: interno vs dejar-una-fuera")

ax[1].scatter(orden["n"], orden["delta_AUROC"], color="#1F3864")
ax[1].axhline(0, color="grey", lw=1, ls="--"); ax[1].set_xscale("log")
ax[1].set_xlabel("Tamano de la entidad (escala log)")
ax[1].set_ylabel("Perdida de AUROC al no ser vista")
ax[1].set_title("Perdida por transportabilidad vs tamano")
plt.tight_layout(); plt.savefig("transportabilidad_loeo.png", dpi=110)
print("Figura guardada: transportabilidad_loeo.png")

salida = {
    "diseno": "dejar-una-entidad-fuera; validacion INTERNA, no externa",
    "n_entidades": int(len(L)),
    "delta_auroc_mediana": round(float(L["delta_AUROC"].median()), 4),
    "delta_auroc_min": round(float(L["delta_AUROC"].min()), 4),
    "delta_auroc_max": round(float(L["delta_AUROC"].max()), 4),
    "n_empeoran": int((L["delta_AUROC"] < 0).sum()),
    "gap_auroc_interno": round(float(L["AUROC_interno"].max() - L["AUROC_interno"].min()), 4),
    "gap_auroc_loeo": round(float(L["AUROC_loeo"].max() - L["AUROC_loeo"].min()), 4),
    "gap_tpr_interno": round(float(L["TPR_interno"].max() - L["TPR_interno"].min()), 4),
    "gap_tpr_loeo": round(float(L["TPR_loeo"].max() - L["TPR_loeo"].min()), 4),
    "corr_perdida_tamano": {"r": round(float(r_n), 3), "p": round(float(p_n), 3)},
    "corr_perdida_tasa": {"r": round(float(r_t), 3), "p": round(float(p_t), 3)},
    "por_entidad": L.round(4).to_dict(orient="records"),
}
with open("transportabilidad_loeo.json", "w", encoding="utf-8") as fh:
    json.dump(salida, fh, indent=2, ensure_ascii=False)
print("Exportado: transportabilidad_loeo.json")

Figura guardada: transportabilidad_loeo.png
Exportado: transportabilidad_loeo.json


## 7. Lectura

Este diseño responde si el modelo y su equidad sobreviven a un estado que no participó en el entrenamiento. Es la evidencia más fuerte de generalización geográfica que puede obtenerse sin una segunda fuente.

Lo que **no** responde: si el modelo transfiere a otro sistema de registro, a otra definición de caso o a otro periodo. Eso exige la base de Egresos Hospitalarios de la DGIS, con el prerrequisito documentado en el capítulo de trabajo futuro: al ser una base de egresos, su población son pacientes hospitalizados, con una prevalencia del desenlace cercana al 48 % frente al 5.6 % de esta cohorte. Validar el modelo actual contra ella confundiría el cambio de espectro con un fallo de transportabilidad.